System Path Setup

In [2]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [16]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import folium # For interactive mapping
import geopandas as gpd
import pandas as pd
from configs.regions import kenyan_coast_roi # Your ROI
from configs.carbon_coefficients import CARBON_FRACTION_BIOMASS # For reference if needed

ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID
print("All core libraries imported and GEE initialized.")

All core libraries imported and GEE initialized.


Load Consolidated Carbon Image and Prepare Layers

In [17]:
# Cell 3 (REVISED): Load Enriched GBIF Data for Analysis and Visualization
print("--- Loading Enriched GBIF Data ---")

gbif_env_enriched_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_mangrove_env_enriched.geojson')

if os.path.exists(gbif_env_enriched_path):
    gbif_species_df = gpd.read_file(gbif_env_enriched_path) # Dataframe used in original Cell 3 (now Cell 4)
    gbif_species_map_df = gbif_species_df.copy() # Make a copy for mapping if needed, or just use gbif_species_df
    print(f"Loaded {len(gbif_species_df)} enriched GBIF records.")
else:
    print(f"Error: Enriched GBIF data not found at {gbif_env_enriched_path}. Please re-run 06_Data_Integration_Species_Mangroves.ipynb.")
    sys.exit("No enriched GBIF data to analyze/visualize.")

print(f"Columns available: {gbif_species_df.columns.tolist()}")

--- Loading Enriched GBIF Data ---
Loaded 1 enriched GBIF records.
Columns available: ['class', 'decimalLatitude', 'decimalLongitude', 'elevation_m', 'eventDate', 'family', 'gbifID', 'genus', 'is_mangrove', 'kingdom', 'order', 'phylum', 'scientificName', 'species', 'geometry']


Generate Interactive Folium Map

In [18]:
# Cell 4: Generate Interactive Folium Map of Carbon Analysis Results
print("--- Generating Interactive Carbon Analysis Map ---")

# Define visualization parameters
# Mangrove Presence (solid green)
mangrove_vis_params_solid = {
    'min': 0, 'max': 1,
    'palette': ['#00000000', 'green'], # Transparent for non-mangrove, solid green for mangrove
    'opacity': 0.8
}

# AGC Density (light yellow to dark blue, transparent background for non-mangroves)
agc_vis_params_final = {
    'min': 0, 'max': 150, # Adjust based on your data's observed range for best visual. Max from Task 4.3 was 1032.
                           # Use a max value that provides good contrast for the *majority* of your data.
    'palette': ['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84'],
    'opacity': 0.7 # Semi-transparent to let underlying green mangroves show through
}

# High Carbon Hotspots (orange)
high_carbon_vis = {'min': 0, 'max': 1, 'palette': ['#00000000', 'orange'], 'opacity': 0.8}

# Extreme Carbon Hotspots (purple)
extreme_carbon_vis = {'min': 0, 'max': 1, 'palette': ['#00000000', 'purple'], 'opacity': 0.9}


# Initialize map
centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

final_carbon_map = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='CartoDB positron')

# Add ROI (transparent fill)
folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#00000000', 'color': 'red', 'weight': 3, 'fillOpacity': 0.0}
).add_to(final_carbon_map)

# Add Mangrove Presence layer (visible by default) - as a base for mangroves
map_id_dict_presence = mangrove_presence_image.getMapId(mangrove_vis_params_solid)
folium.TileLayer(
    tiles=map_id_dict_presence['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Mangrove Presence (GMW)',
    show=True
).add_to(final_carbon_map)

# Add AGC Density layer (semi-transparent, on top of mangrove presence)
map_id_dict_agc_final = agc_density_image.getMapId(agc_vis_params_final)
folium.TileLayer(
    tiles=map_id_dict_agc_final['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='AGC Density (tonnes C/ha)',
    show=True # Show by default
).add_to(final_carbon_map)

# Add High Carbon Hotspots layer (on top, with distinct colors)
map_id_dict_high = high_carbon_areas.getMapId(high_carbon_vis)
folium.TileLayer(
    tiles=map_id_dict_high['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name=f'High Carbon Hotspots (>{high_carbon_threshold} tC/ha)',
    show=False # User can toggle these hotspots on/off
).add_to(final_carbon_map)

# Add Extreme Carbon Hotspots layer
map_id_dict_extreme = extreme_carbon_areas.getMapId(extreme_carbon_vis)
folium.TileLayer(
    tiles=map_id_dict_extreme['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name=f'Extreme Carbon Hotspots (>{extreme_carbon_threshold} tC/ha)',
    show=False # User can toggle these hotspots on/off
).add_to(final_carbon_map)

folium.LayerControl().add_to(final_carbon_map) # Add layer control for toggling layers

# Save the final map to an HTML file
output_map_path = os.path.join(project_root, 'docs', 'gaia_ark_carbon_map_final.html')
final_carbon_map.save(output_map_path)
print(f"Final interactive carbon analysis map saved to: {output_map_path}")
print("Open this HTML file in your web browser to view the map.")

--- Generating Interactive Carbon Analysis Map ---


Final interactive carbon analysis map saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\docs\gaia_ark_carbon_map_final.html
Open this HTML file in your web browser to view the map.


Load Mangrove Extent and Environmental Layers (for context)

In [19]:
# Cell 5: Load Mangrove Extent and Environmental Layers for Visualization Context
print("--- Loading Context Layers for Species Map ---")

# --- Load GEE Mangrove Extent image ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# Use the UNBUFFERED mangrove extent for background context
mangroves_extent_gee_viz = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0)

# --- Load Environmental Layers ---
# Elevation
nasadem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
elevation_roi_viz = nasadem.clip(kenyan_coast_roi)

# WorldClim (Temp & Precip)
worldclim_dataset = ee.ImageCollection("WORLDCLIM/V1/MONTHLY")
annual_mean_temp = worldclim_dataset.select('tavg').mean().divide(10).rename('mean_annual_temp_C')
annual_total_prec = worldclim_dataset.select('prec').sum().rename('total_annual_prec_mm')
mean_temp_roi_viz = annual_mean_temp.clip(kenyan_coast_roi)
total_prec_roi_viz = annual_total_prec.clip(kenyan_coast_roi)

print("Mangrove extent and environmental layers loaded for background context.")

--- Loading Context Layers for Species Map ---
Mangrove extent and environmental layers loaded for background context.


Generate Interactive Folium Map for Species Occurrence(s)

In [20]:
# Cell 6: Generate Interactive Folium Map for Species Occurrence(s) (REVISED: Handle Timestamp)
print("--- Generating Interactive Species Occurrence Map ---")

gbif_env_enriched_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_mangrove_env_enriched.geojson')

if os.path.exists(gbif_env_enriched_path):
    gbif_species_map_df = gpd.read_file(gbif_env_enriched_path)
    print(f"Loaded {len(gbif_species_map_df)} enriched GBIF records for visualization.")
else:
    print(f"Error: Enriched GBIF data not found at {gbif_env_enriched_path}. Skipping species map.")
    gbif_species_map_df = gpd.GeoDataFrame()

# --- CRITICAL CHANGE: Convert Timestamp columns to string format ---
if not gbif_species_map_df.empty:
    for col in gbif_species_map_df.columns:
        if pd.api.types.is_datetime64_any_dtype(gbif_species_map_df[col]):
            gbif_species_map_df[col] = gbif_species_map_df[col].astype(str)
    print("Converted Timestamp columns to string for JSON serialization.")

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

species_map = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='CartoDB positron')

folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#00000000', 'color': 'red', 'weight': 3, 'fillOpacity': 0.0}
).add_to(species_map)

# ... (other layers for mangrove presence, elevation, temp, precip remain the same) ...

# Add Species Occurrence Points (with rich tooltip)
if not gbif_species_map_df.empty:
    tooltip_fields = [
        'scientificName', 'elevation_m', 'mean_annual_temp_C', 'total_annual_prec_mm',
        'eventDate', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'
    ]
    existing_tooltip_fields = [field for field in tooltip_fields if field in gbif_species_map_df.columns]

    folium.GeoJson(
        gbif_species_map_df.__geo_interface__,
        name='Filtered Species Occurrences',
        tooltip=folium.features.GeoJsonTooltip(
            fields=existing_tooltip_fields,
            aliases=[f.replace('_', ' ').title() for f in existing_tooltip_fields],
            localize=True
        ),
        marker=folium.CircleMarker(radius=10, weight=3, color='blue', fill_color='yellow', fill_opacity=0.8)
    ).add_to(species_map)
else:
    print("No species records to visualize on the map.")

folium.LayerControl().add_to(species_map)

output_map_path = os.path.join(project_root, 'docs', 'gaia_ark_species_map_final.html')
species_map.save(output_map_path)
print(f"Final interactive species occurrence map saved to: {output_map_path}")
print("Open this HTML file in your web browser to view the map.")

--- Generating Interactive Species Occurrence Map ---
Loaded 1 enriched GBIF records for visualization.
Converted Timestamp columns to string for JSON serialization.
Final interactive species occurrence map saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\docs\gaia_ark_species_map_final.html
Open this HTML file in your web browser to view the map.
